# Prepare Town Neck Beach RGB cells

Independent external-location dataset from Dryad Dataset A (DOI: 10.5061/dryad.2fqz612ps). Original files are read only. Outputs are separate from the Svalbard data and splits.

The 5 cm RGB and colour-label GeoTIFF tiles are assembled on their shared EPSG:26919 grid before extracting 1 m square (20 x 20 pixel) cells. The 25 cm duplicate imagery is not used. Dense Doodler labels are generated from sparse human annotations; derived cells are not independent manual annotations.

In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'Source').is_dir() and (p / 'Notebooks').is_dir())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

SOURCE_ROOT = Path.home() / 'Downloads' / 'datasetA'
OUTPUT_ROOT = PROJECT_ROOT / 'Data' / 'Raw' / 'bs_town_neck'
PURITY_THRESHOLD = 0.90
PROGRESS_EVERY = 10000

from Source.bs_town_neck_preprocessing import prepare_dataset
print(f'Source: {SOURCE_ROOT}\nOutput: {OUTPUT_ROOT}', flush=True)

Source: /Users/piotr/Downloads/datasetA
Output: /Users/piotr/Praca/Nauka/publikacje/Context_Aware_Deep_Learning_Beach_Classification/software/Data/Raw/bs_town_neck


## Selection and output policy

A labelled target requires 400 valid RGB pixels, 400 pixels labelled among classes 1-6, and at least 360 pixels of one class. Exclude black/unlabelled labels, nodata (7), unusual (8), and mixed cells below the threshold. All-black RGB pixels are marked unavailable and exported as NaN; no normalization or mean replacement is applied before splitting.

Export every grid cell containing any available RGB as context, including cells excluded as labelled targets. Keep the six original substantive class codes, with separate Classes.xlsx. Descriptor_bs_town_neck.csv contains only date, cell_id, class. The inventory records every complete grid cell's coordinates, validity, purity, selection, and exclusion reason.

IDs use column * grid_rows + row + 1, with zero-based rows increasing south and columns increasing east. Grid dimensions and georeferencing are recorded in bs_town_neck_grid_metadata.json. Partial outermost strips smaller than a full cell are omitted.

**This notebook does not split data or train models.** A later Town Neck split must use spatial blocks and non-overlapping context footprints (up to radius 3 if tested). Do not use the Svalbard survey-date splitter, class mapping, fixed grid dimensions, or normalization statistics. Parameterize the existing context loader and split helpers with this local grid before training. Elevation and slope are not included; use RGB_only unless matching topography is obtained separately.

A completed dataset is protected from overwriting. An interrupted conversion may be rerun with the same settings. Dependencies are numpy, Pillow (installed with torchvision), and openpyxl.

In [2]:
import json
import hashlib
summary_path = OUTPUT_ROOT / 'bs_town_neck_preprocessing_summary.json'
if summary_path.exists():
    summary = json.loads(summary_path.read_text())
    grid_metadata = json.loads((OUTPUT_ROOT / 'bs_town_neck_grid_metadata.json').read_text())
    if grid_metadata['purity_threshold'] != PURITY_THRESHOLD:
        raise ValueError('Saved purity threshold differs; choose a new OUTPUT_ROOT to change selection.')
    if (OUTPUT_ROOT / 'bs_town_neck_preprocessing_in_progress.json').exists():
        raise ValueError('Conflicting completed and interrupted-run markers; verify the output directory.')
    for filename, key in [('Descriptor_bs_town_neck.csv', 'descriptor_sha256'), ('bs_town_neck_cell_inventory.csv', 'inventory_sha256')]:
        if hashlib.sha256((OUTPUT_ROOT / filename).read_bytes()).hexdigest() != summary[key]:
            raise ValueError(f'Saved file changed: {filename}')
    for entry in grid_metadata['source_manifest']:
        if hashlib.sha256((SOURCE_ROOT / entry['path']).read_bytes()).hexdigest() != entry['sha256']:
            raise ValueError(f"Source file changed: {entry['path']}")
    print('Dataset already prepared; no files overwritten.', flush=True)
    print(json.dumps(summary, indent=2), flush=True)
else:
    summary = prepare_dataset(SOURCE_ROOT, OUTPUT_ROOT, PURITY_THRESHOLD, PROGRESS_EVERY)

Dataset already prepared; no files overwritten.


{
  "grid_cells": 253786,
  "incomplete_rgb": 1946,
  "context_cells": 253176,
  "selected_cells": 233343,
  "mixed_classes": 17267,
  "invalid_or_ambiguous_labels": 1230,
  "matrix_files": 759528,
  "selected_cells_by_class": {
    "Water": 52310,
    "Sand": 56363,
    "Gravel": 3239,
    "Cobble/boulder": 1380,
    "Vegetated": 116922,
    "Development": 3129
  },
  "elapsed_seconds": 137.58393587497994,
  "descriptor_sha256": "c43ed6f8b7b9b2a5c5d69309fa5e8bf86fc2b30daa347836f8339513e72ac7a8",
  "inventory_sha256": "35d861350857d172eb0a6119aaae57eec541e08e5188943cde103b514bf39a7d"
}


In [3]:
import pandas as pd
descriptor = pd.read_csv(OUTPUT_ROOT / 'Descriptor_bs_town_neck.csv', dtype={'date': str, 'cell_id': str})
assert list(descriptor.columns) == ['date', 'cell_id', 'class']
assert not descriptor.duplicated(['date', 'cell_id']).any()
assert set(descriptor['class']) == set(range(1, 7))
assert len(descriptor) == summary['selected_cells']
display(descriptor.groupby('class').size().rename('Selected cells').to_frame())
grid = json.loads((OUTPUT_ROOT / 'bs_town_neck_grid_metadata.json').read_text())
print(f"Grid: {grid['grid_rows']} rows x {grid['grid_columns']} columns; channels: RGB only.", flush=True)

,Selected cells
class,
1,52310
2,56363
3,3239
4,1380
5,116922
6,3129


Grid: 559 rows x 454 columns; channels: RGB only.
